# Task 1 — Header Row Detection

Client ERP exports often put a **title banner** (and blank rows) *above* the real column names.  
Hardcoding `header=0` silently breaks the whole pipeline.

### Scoring formula (per candidate row `i`)
```
score = string_density
      + type_consistency_below(i)
      + 1.25 × label_score(i)
      − sparsity_penalty(i)
      + tiny early-row bonus
```
Pick `argmax(score)` in the first ~15 rows. Caller **confirms** before load.

### Why this approach
| Option | Verdict |
|---|---|
| Always `header=0` | Fails on title rows |
| openpyxl cell-style scan | Extra complexity; styles unreliable on `.xls` |
| **Custom pandas heuristic** | Accurate enough, explainable, works on `.xls`/`.xlsx`/CSV |

Libs: `pandas`, `openpyxl` / `xlrd` (engine by extension).

In [6]:
import re
from pathlib import Path
from typing import Any

import pandas as pd

MAX_SCAN_ROWS = 15
LOOKAHEAD = 5

HEADER_TOKEN_RE = re.compile(
    r"(name|code|date|desc|qty|quantity|address|customer|supplier|product|"
    r"ref|amount|price|cost|site|order|invoice|phone|fax|country|city|post|"
    r"number|line|status|currency|margin|stock|lot|pack|value|terms|vat)",
    re.I,
)
ID_LIKE_RE = re.compile(r"^[A-Z]{1,6}\d{3,}[A-Z0-9-]*$", re.I)
NUMERIC_LIKE_RE = re.compile(r"^-?\d+([.,]\d+)?$")


def _is_empty(value: Any) -> bool:
    return value is None or (isinstance(value, float) and pd.isna(value)) or str(value).strip() == ""


def _is_stringy(value: Any) -> bool:
    return isinstance(value, str) and bool(value.strip())


def string_density(row: pd.Series) -> float:
    vals = [v for v in row.tolist() if not _is_empty(v)]
    if not vals:
        return 0.0
    return sum(1 for v in vals if _is_stringy(v)) / len(vals)


def type_consistency_below(raw_df: pd.DataFrame, header_idx: int, lookahead: int = LOOKAHEAD) -> float:
    """Do rows under this candidate look typed/stable? Good headers sit above consistent data."""
    start, end = header_idx + 1, min(header_idx + 1 + lookahead, len(raw_df))
    if start >= end:
        return 0.0
    block = raw_df.iloc[start:end]
    scores = []
    for col in block.columns:
        values = [v for v in block[col].tolist() if not _is_empty(v)]
        if len(values) < 2:
            scores.append(0.5 if values else 0.0)
            continue
        kinds = []
        for v in values:
            if isinstance(v, bool):
                kinds.append("bool")
            elif isinstance(v, (int, float)):
                kinds.append("number")
            elif isinstance(v, str):
                kinds.append("string")
            else:
                kinds.append(type(v).__name__)
        scores.append(max(kinds.count(k) for k in set(kinds)) / len(kinds))
    return sum(scores) / len(scores) if scores else 0.0


def label_score(row: pd.Series) -> float:
    """Reward column-name tokens; punish ID/number-looking cells (those are data, not headers)."""
    vals = [str(v).strip() for v in row.tolist() if not _is_empty(v)]
    if not vals:
        return 0.0
    token_hits = sum(1 for v in vals if HEADER_TOKEN_RE.search(v))
    id_like = sum(1 for v in vals if ID_LIKE_RE.match(v) or NUMERIC_LIKE_RE.match(v))
    unique_ratio = len(set(vals)) / len(vals)
    return (token_hits / len(vals)) + 0.25 * unique_ratio - (id_like / len(vals))


def sparsity_penalty(row: pd.Series, n_cols: int) -> float:
    vals = [str(v).strip() for v in row.tolist() if not _is_empty(v)]
    nn = len(vals)
    if nn < max(3, n_cols // 3):
        penalty = 1.5
    elif nn < max(3, n_cols // 2):
        penalty = 0.5
    else:
        penalty = 0.0
    # Long banner titles in 1–3 cells ("Client Export - Q1 2024")
    if nn and (sum(len(v) for v in vals) / nn) > 45 and nn <= 3:
        penalty += 1.0
    return penalty


def score_row(raw_df: pd.DataFrame, i: int, scan: int) -> dict[str, float]:
    row = raw_df.iloc[i]
    density = string_density(row)
    consistency = type_consistency_below(raw_df, i)
    labels = label_score(row)
    penalty = sparsity_penalty(row, raw_df.shape[1])
    early = max(0.0, (scan - i) * 0.001)
    total = density + consistency + 1.25 * labels - penalty + early
    return {
        "row": i,
        "density": round(density, 4),
        "consistency": round(consistency, 4),
        "label": round(labels, 4),
        "penalty": round(penalty, 4),
        "early": round(early, 4),
        "total": round(total, 4),
    }


def detect_header_row(raw_df: pd.DataFrame, max_scan_rows: int = MAX_SCAN_ROWS) -> int:
    if raw_df is None or raw_df.empty:
        raise ValueError("empty dataframe")
    scan = min(max_scan_rows, len(raw_df))
    best_idx, best_score = 0, float("-inf")
    for i in range(scan):
        s = score_row(raw_df, i, scan)["total"]
        if s > best_score:
            best_score, best_idx = s, i
    return int(best_idx)


def explain_detection(raw_df: pd.DataFrame, max_scan_rows: int = MAX_SCAN_ROWS) -> pd.DataFrame:
    """Full scoreboard — makes the choice inspectable (like Q1/Q3 for outliers)."""
    scan = min(max_scan_rows, len(raw_df))
    rows = [score_row(raw_df, i, scan) for i in range(scan)]
    board = pd.DataFrame(rows)
    board["preview"] = [
        " | ".join(str(v) for v in raw_df.iloc[i].tolist()[:4] if not _is_empty(v))[:60]
        for i in range(scan)
    ]
    return board.sort_values("total", ascending=False).reset_index(drop=True)


def merge_multirow_header(raw_df: pd.DataFrame, header_row: int, parent_rows: int = 1) -> list[str]:
    headers = raw_df.iloc[header_row].tolist()
    parents = raw_df.iloc[header_row - parent_rows].tolist() if header_row - parent_rows >= 0 else None
    merged = []
    for i, h in enumerate(headers):
        child = "" if _is_empty(h) else str(h).strip()
        parent = ""
        if parents is not None and not _is_empty(parents[i]):
            parent = str(parents[i]).strip()
        if parent and child and parent.lower() != child.lower():
            merged.append(f"{parent} {child}")
        else:
            merged.append(child or parent or f"unnamed_{i}")
    return merged


def _unique_names(names: list[str]) -> list[str]:
    out, seen = [], {}
    for i, name in enumerate(names):
        base = name.strip() if name and str(name).strip() else f"unnamed_{i}"
        if base in seen:
            seen[base] += 1
            out.append(f"{base}_{seen[base]}")
        else:
            seen[base] = 0
            out.append(base)
    return out


def load_with_confirmed_header(
    raw_df: pd.DataFrame,
    header_row: int,
    *,
    merge_parent: bool = True,
) -> pd.DataFrame:
    if merge_parent and header_row > 0 and string_density(raw_df.iloc[header_row - 1]) >= 0.4:
        names = merge_multirow_header(raw_df, header_row)
    else:
        names = [
            f"unnamed_{i}" if _is_empty(h) else str(h).strip()
            for i, h in enumerate(raw_df.iloc[header_row].tolist())
        ]
    body = raw_df.iloc[header_row + 1 :].copy()
    body.columns = _unique_names(names)
    return body.dropna(how="all").reset_index(drop=True)


def read_excel_raw(filepath: str | Path) -> pd.DataFrame:
    path = Path(filepath)
    engine = "xlrd" if path.suffix.lower() == ".xls" else "openpyxl"
    sheets = pd.read_excel(path, sheet_name=None, header=None, engine=engine, dtype=object)
    return next(iter(sheets.values()))


print("Task 1 helpers ready (detect + explain + load)")

Task 1 helpers ready (detect + explain + load)


## Case A — Title banner + blank row (classic failure of `header=0`)

In [7]:
raw_a = pd.DataFrame([
    ["Client Export - Q1 2024", None, None, None],
    [None, None, None, None],
    ["Customer Name", "Customer Code", "City", "Amount"],
    ["Ali Traders", "C001", "Lahore", 1200],
    ["Sara Mills", "C002", "Karachi", 850],
    ["Omar Co", "C003", "Islamabad", 410],
])
print("Raw preview:")
display(raw_a)

print("\nScoreboard (highest total wins):")
board = explain_detection(raw_a)
display(board)

h = detect_header_row(raw_a)
print(f"Detected header row: {h} → {raw_a.iloc[h].tolist()}")
print(f"Naive header=0 would wrongly use: {raw_a.iloc[0].tolist()}")

df_a = load_with_confirmed_header(raw_a, h, merge_parent=False)
print("\nClean frame:")
display(df_a)

Raw preview:


,0,1,2,3
0,Client Export - Q1 2024,NaN,NaN,None
1,NaN,NaN,NaN,None
2,Customer Name,Customer Code,City,Amount
3,Ali Traders,C001,Lahore,1200
4,Sara Mills,C002,Karachi,850
5,Omar Co,C003,Islamabad,410



Scoreboard (highest total wins):


,row,density,consistency,label,penalty,early,total,preview
0,2,1.00,1.0000,1.25,0.0,0.004,3.5665,Customer Name | Customer Code | City | Amount
1,3,0.75,1.0000,-0.25,0.0,0.003,1.4405,Ali Traders | C001 | Lahore | 1200
2,4,0.75,0.5000,-0.25,0.0,0.002,0.9395,Sara Mills | C002 | Karachi | 850
3,0,1.00,0.9375,0.25,1.5,0.006,0.7560,Client Export - Q1 2024
4,5,0.75,0.0000,-0.25,0.0,0.001,0.4385,Omar Co | C003 | Islamabad | 410
5,1,0.00,0.9375,0.00,1.5,0.005,-0.5575,


Detected header row: 2 → ['Customer Name', 'Customer Code', 'City', 'Amount']
Naive header=0 would wrongly use: ['Client Export - Q1 2024', nan, nan, None]

Clean frame:


,Customer Name,Customer Code,City,Amount
0,Ali Traders,C001,Lahore,1200
1,Sara Mills,C002,Karachi,850
2,Omar Co,C003,Islamabad,410


## Case B — Multi-row header (parent group labels + real columns)
Similar to Booked Orders style exports.

In [8]:
raw_b = pd.DataFrame([
    ["Order Info", "Order Info", "Product", "Product", "Totals"],
    ["Order No", "Order Date", "Product Code", "Qty", "Line Value"],
    ["SO1001", "2024-01-05", "P55", 10, 250.0],
    ["SO1002", "2024-01-06", "P77", 3, 90.0],
])
display(raw_b)

h = detect_header_row(raw_b)
print("Detected header row:", h)
print("Scoreboard:")
display(explain_detection(raw_b))

df_b = load_with_confirmed_header(raw_b, h, merge_parent=True)
print("Merged column names:", list(df_b.columns))
display(df_b)

,0,1,2,3,4
0,Order Info,Order Info,Product,Product,Totals
1,Order No,Order Date,Product Code,Qty,Line Value
2,SO1001,2024-01-05,P55,10,250.0
3,SO1002,2024-01-06,P77,3,90.0


Detected header row: 1
Scoreboard:


,row,density,consistency,label,penalty,early,total,preview
0,1,1.0,1.0000,1.25,0.0,0.003,3.5655,Order No | Order Date | Product Code | Qty
1,0,1.0,0.8667,0.95,0.0,0.004,3.0582,Order Info | Order Info | Product | Product
2,2,0.6,0.5000,-0.35,0.0,0.002,0.6645,SO1001 | 2024-01-05 | P55 | 10
3,3,0.6,0.0000,-0.35,0.0,0.001,0.1635,SO1002 | 2024-01-06 | P77 | 3


Merged column names: ['Order Info Order No', 'Order Info Order Date', 'Product Product Code', 'Product Qty', 'Totals Line Value']


,Order Info Order No,Order Info Order Date,Product Product Code,Product Qty,Totals Line Value
0,SO1001,2024-01-05,P55,10,250.0
1,SO1002,2024-01-06,P77,3,90.0


## Edge cases

In [9]:
cases = {
    "already_clean": pd.DataFrame([
        ["Name", "Age", "City"],
        ["Ali", 30, "Lahore"],
        ["Sara", 25, "Karachi"],
    ]),
    "deep_title": pd.DataFrame([
        ["CONFIDENTIAL", None, None],
        ["Prepared for audit", None, None],
        [None, None, None],
        ["Code", "Description", "Qty"],
        ["A1", "Widget", 2],
    ]),
    "data_lookalike": pd.DataFrame([
        ["C1001", "C1002", "C1003"],   # IDs — should NOT win as header
        ["Name", "Code", "City"],
        ["Ali", "C1001", "Lahore"],
    ]),
}

for name, raw in cases.items():
    h = detect_header_row(raw)
    top = explain_detection(raw).iloc[0]
    print(f"{name:16} → header={h}  values={raw.iloc[h].tolist()}  top_score={top['total']}")

already_clean    → header=0  values=['Name', 'Age', 'City']  top_score=3.1488
deep_title       → header=3  values=['Code', 'Description', 'Qty']  top_score=3.0645
data_lookalike   → header=1  values=['Name', 'Code', 'City']  top_score=3.0645


## Testing on dataset (skip if folder missing)

In [10]:
dataset = Path("OneDrive_1_26-01-2026 - latest data set")
if not dataset.exists():
    dataset = Path("../OneDrive_1_26-01-2026 - latest data set")

path = dataset / "Customer List.xls"
if path.exists():
    raw = read_excel_raw(path)
    print("file:", path.name, "| raw shape:", raw.shape)
    print("first 4 raw rows:")
    display(raw.head(4))
    print("scoreboard:")
    display(explain_detection(raw).head(6))
    h = detect_header_row(raw)
    df = load_with_confirmed_header(raw, h)
    print("detected header:", h)
    print("columns:", list(df.columns)[:10])
    print("shape:", df.shape)
    display(df.head(3))
else:
    print("Teacher dataset not found — synthetic cases above are enough.")

file: Customer List.xls | raw shape: (537, 30)
first 4 raw rows:


,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,Customer List as at 20/01/2026; Active Custom...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Customer No.,Company Name,Default?,Add. Code,Address Line 1,Address Line 2,Address Line 3,City,Post Code,Country,...,External Rep,Internal Rep,Ship Address VAT Number,Customer VAT Number,Customer EORI,Company Reg No.,Credit Limit,Overall Credit Limit,Pay Terms,Custom Hold Status


scoreboard:


,row,density,consistency,label,penalty,early,total,preview
0,3,1.0000,0.8000,0.8500,0.0,0.012,2.8745,Customer No. | Company Name | Default? | Add. ...
1,2,1.0000,0.9367,1.2500,1.5,0.013,2.0122,
2,14,0.9231,0.9333,0.0769,0.0,0.001,1.9536,C00002 | Eaton Electric Limited | No | DEL1
3,13,0.9130,0.9333,-0.0761,0.0,0.002,1.7533,C00002 | Eaton Electric Limited | Yes | HQ
4,10,0.8947,0.9333,-0.0789,0.0,0.005,1.7344,C00001 | Easby Electronics Ltd | No | TYCO
5,12,0.8889,0.9333,-0.0972,0.0,0.003,1.7037,C00001 | Easby Electronics Ltd | No | WH


detected header: 3
columns: ['Customer No.', 'Company Name', 'Default?', 'Add. Code', 'Address Line 1', 'Address Line 2', 'Address Line 3', 'City', 'Post Code', 'Country']
shape: (533, 30)


,Customer No.,Company Name,Default?,Add. Code,Address Line 1,Address Line 2,Address Line 3,City,Post Code,Country,...,External Rep,Internal Rep,Ship Address VAT Number,Customer VAT Number,Customer EORI,Company Reg No.,Credit Limit,Overall Credit Limit,Pay Terms,Custom Hold Status
0,C00001,Easby Electronics Ltd,Yes,HQ,4 Bailey Court,Chartermark Way,Colburn Business Park,CATTERICK,DL9 4QL,United Kingdom,...,Unassigned,Lorraine Bonner,NaN,GB329531846,GB329531846,01537952,300000,1000000,30EOM,Strategic Account – Never goes on hold
1,C00001,Easby Electronics Ltd,No,CIRCA,Unit 10,Penny Corner,Farthing Rd,IPSWICH,IP1 5AP,United Kingdom,...,Unassigned,Lorraine Bonner,NaN,GB329531846,GB329531846,01537952,300000,1000000,30EOM,Strategic Account – Never goes on hold
2,C00001,Easby Electronics Ltd,No,JOHNS,"SPECTOR HOUSE, DABELL AVENUE",BLENHEIM INDUSTRIAL ESTATE,NaN,BULWELL,NG6 8WA,United Kingdom,...,Unassigned,Lorraine Bonner,NaN,GB329531846,GB329531846,01537952,300000,1000000,30EOM,Strategic Account – Never goes on hold


## Quick comparison

| Approach | Title banners | Multi-row headers | Explainable scores | Phase 1 choice |
|---|---|---|---|---|
| Hardcode `header=0` | fails | fails | no | no |
| openpyxl style scan | mixed | weak on `.xls` | low | no |
| **Custom scoreboard** | works | merge parent row | yes | **YES** |

## Human-in-the-Loop Principle
Pipeline rule: detect → show preview → **user confirms** → load.  
Same logic lives in `data_quality_engine/engine/ingestion.py`.